# Lab 3: inductive biases in the pipeline

Lab 2 left a working pipeline and a Vision Transformer that beats persistence at six hours.
This lab changes that pipeline one inductive bias at a time, along the listings of the Day 3 lecture, Part 4: the weights in the loss (part 1), the roll-out in the training step (part 2), and one embedding per variable with a weight per variable in the loss (part 3).
Every part runs the same five steps: (1) implement the module and incorporate it into your pipeline; (2) train a model with it; (3) plot a sample; (4) plot the skill score against climatology, with the models of the earlier parts on the same axes; (5) plot one task-specific metric of your choice.
Section 0 sets up what the parts need: a trained model saved as a checkpoint, its forecasts of the evaluation year as an xarray Dataset in the schema of lab 1, the sample plots, and the skill score.
An extension at the end gives the model the time of day and the season.

The training runs are longer than in lab 2 (2000 steps in the given configurations, a minute or two with a GPU).
On a laptop without one, lower `max_steps` in the configuration files; the comparisons hold at fewer steps.
Conventions as in lab 2: tensors carry the axes `(batch, variable, time, latitude, longitude)`, every reshape is a named einops pattern, every setting is a config field, and each piece goes into its module under `utils/`.

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import torch
import lightning as L
from lightning.pytorch.loggers import CSVLogger

xr.set_options(keep_attrs=True, display_expand_data=False, use_bottleneck=False)
torch.manual_seed(0)

### Documentation

- [Checkpoints](https://lightning.ai/docs/pytorch/stable/common/checkpointing_basic.html): `Trainer.save_checkpoint` and `LightningModule.load_from_checkpoint`.
- [Trainer.predict](https://lightning.ai/docs/pytorch/stable/deploy/production_basic.html): `predict_step` and `predict_dataloader`.
- [Callbacks](https://lightning.ai/docs/pytorch/stable/extensions/callbacks.html): the hooks a `Callback` can fill, `on_train_batch_end`, `on_validation_end`, `on_train_end`.
- [CSVLogger](https://lightning.ai/docs/pytorch/stable/extensions/generated/lightning.pytorch.loggers.CSVLogger.html): where `self.log` values go when the Trainer has a logger.
- [torch.utils.data.Subset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Subset): a Dataset restricted to a list of indices.
- [xarray vectorised indexing](https://docs.xarray.dev/en/stable/user-guide/indexing.html#vectorized-indexing): `.sel` with DataArray indexers, the alignment of forecast and truth from lab 1.

### Data on disk

The two stores of lab 2, the fields and their statistics, plus one more: the climatology of the training years, the mean per hour of day and day of year over 2015 to 2018, as in lab 1, section 2.
The cell writes it once from the fields on disk; the climatology forecast of section 0 and the skill scores read from it.

In [7]:
DATA = Path("data")
ZARR = DATA / "era5_5p6.zarr"       # written by the setup cell of lab 2
STATS = DATA / "stats.zarr"
CLIM = DATA / "clim_5p6.zarr"
TRAIN = slice("2015", "2018")

if not CLIM.exists():
    fields = xr.open_zarr(ZARR).sel(time=TRAIN).load()
    clim = fields.groupby(["time.hour", "time.dayofyear"]).mean("time").transpose("hour", "dayofyear", "latitude", "longitude")
    clim.to_zarr(CLIM, mode="w", zarr_format=2)

era5 = xr.open_zarr(ZARR)           # the fields, lazily: the truth is aligned from here
clim = xr.open_zarr(CLIM).load()    # 108 MB in memory
clim

<xarray.Dataset> Size: 108MB
Dimensions:    (hour: 4, dayofyear: 366, latitude: 32, longitude: 64)
Coordinates:
  * hour       (hour) int64 32B 0 6 12 18
  * dayofyear  (dayofyear) int64 3kB 1 2 3 4 5 6 7 ... 361 362 363 364 365 366
  * latitude   (latitude) float64 256B -87.19 -81.56 -75.94 ... 81.56 87.19
  * longitude  (longitude) float64 512B 0.0 5.625 11.25 ... 343.1 348.8 354.4
Data variables:
    Q700       (hour, dayofyear, latitude, longitude) float32 12MB 0.000429 ....
    T2M        (hour, dayofyear, latitude, longitude) float32 12MB 246.8 ... ...
    T850       (hour, dayofyear, latitude, longitude) float32 12MB 256.3 ... ...
    TP6h       (hour, dayofyear, latitude, longitude) float32 12MB 5.198e-06 ...
    U10M       (hour, dayofyear, latitude, longitude) float32 12MB -2.964 ......
    U250       (hour, dayofyear, latitude, longitude) float32 12MB 0.1554 ......
    V10M       (hour, dayofyear, latitude, longitude) float32 12MB -4.487 ......
    V250       (hour, dayofyear, latitude, longitude) float32 12MB -5.764 ......
    Z500       (hour, dayofyear, latitude, longitude) float32 12MB 5.023e+04 ...

In [8]:
# where the runs of this lab leave their traces; every run has a name, used for its config file, log, checkpoint, and forecasts
LOGS = Path("logs")
CHECKPOINTS = Path("checkpoints")
FORECASTS = DATA / "forecasts"
for p in (LOGS, CHECKPOINTS, FORECASTS):
    p.mkdir(parents=True, exist_ok=True)

SKILL = {}                          # run name -> skill score against climatology, filled in section 0.4 and every part after it

## 0. A trained model, its checkpoint, and its forecasts

### 0.1 Train, save, load

The configuration files of this lab (`configs/vit_*.yaml`) carry the fields of every section, so add them to the dataclasses first, each with a default that keeps lab 2's behaviour: `predict_steps` and `predict_stride` in `TrainerConfig` (section 0.2), `train_rollout_steps` and `pre_steps` in `TrainerConfig` (part 2), `separable_embed`, `metadata_embed`, and `dim_metadata` in `NetworkConfig` (part 3 and the extension).
The sections give them their meaning; until then they are unused.

`configs/vit_mse.yaml` is the lab 2 model, trained for 2000 steps, with the prediction settings of section 0.2.
The Trainer below differs from lab 2 in two ways.
It has a logger, `CSVLogger`, so every value passed to `self.log` lands in `logs/<run>/version_<k>/metrics.csv`, one row per logged step; part 1 reads the loss curves from there.
And it saves the trained weights.
`trainer.save_checkpoint(path)` writes one file with the `state_dict` of the module, the optimiser and scheduler states, the step count, and the hyperparameters stored by lab 2's `save_hyperparameters(config.to_dict())`.
A run can then be resumed, fine-tuned, or evaluated later without training again.
Lab 2's `enable_checkpointing=False` switched off the automatic form, the `ModelCheckpoint` callback, which writes the same file on a schedule; the explicit call after `fit` is enough here.

In [ ]:
from utils.config import Config
from utils.lightning_module import ForecastModule

RUN = "vit_mse"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=200, limit_val_batches=20)
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
print(trainer.logger.log_dir, {k: round(float(v), 4) for k, v in trainer.callback_metrics.items()})

Loading the checkpoint back: `torch.load` shows what the file holds, and `ForecastModule.load_from_checkpoint` builds the module and loads the weights.
Our module takes one argument, `config`, while `save_hyperparameters(config.to_dict())` stored the plain dictionary, so the config is rebuilt from `ckpt["hyper_parameters"]` and passed in; a module that stores the config itself as its hyperparameter would not need the extra line.

In [ ]:
path = CHECKPOINTS / f"{RUN}.ckpt"
ckpt = torch.load(path, map_location="cpu", weights_only=False)
print(list(ckpt), "| global_step:", ckpt["global_step"], "| hyper_parameters:", list(ckpt["hyper_parameters"]))

config = Config.from_dict(ckpt["hyper_parameters"])
module = ForecastModule.load_from_checkpoint(path, config=config)

### 0.2 Forecasts of the evaluation year as xarray

Lab 2, section 7, rolled a handful of states out by hand.
Here the roll-out is a third step of the LightningModule, next to the training and validation steps, and `Trainer.predict` runs it over the evaluation year; the forecasts are then converted with the dataset's `to_xarray` into the schema of lab 1 and written to disk once per run.

- `predict_step(batch, batch_idx)`: rolls the model out from `batch[:, :, 0]` for `predict_steps` steps with `forecast` and returns the result on the CPU, `(b, v, predict_steps, H, W)`.
- `predict_dataloader()`: a `DataLoader`, unshuffled, over every `predict_stride`-th window of the validation dataset whose `predict_steps` targets still lie inside the validation period.
  The windows are a `torch.utils.data.Subset` of the validation dataset with the index range `range(0, time - predict_steps, predict_stride)`.
  With `predict_stride = 5` on six-hourly data the initialisations are 30 hours apart, so they run through 00, 06, 12, and 18 UTC in turn, 72 of each over the year; the extension at the end needs all four hours.
- `TrainerConfig` gains `predict_steps` (20: five days) and `predict_stride` (5).

The conversion below is given: it is the evaluation flow of the lecturer's research code, adapted to this pipeline.
`Trainer.predict` returns one tensor per batch.
Every batch is converted with `to_xarray`, with its initialisation times and the lead times as the two named axes; lab 2's `to_xarray` takes its middle axes from the keyword arguments, so the two go in together, and a version that accepts one named axis only is extended to several.
The batches are concatenated along the initialisation time, and the Dataset is written to a zarr store and reopened from it, so that the forecasts of a run stay on disk.
Put the function into `utils/lightning_module.py` next to the module.

In [ ]:
from einops import rearrange

def forecasts_to_xarray(module, predictions, path=None):
    '''The list Trainer.predict returns as one Dataset in the forecast schema of lab 1: (time, prediction_timedelta, latitude, longitude), physical units.'''
    dataset, cfg = module.val_dataset, module.config.trainer
    times = dataset.dataset.time.values
    inits = times[range(0, len(times) - cfg.predict_steps, cfg.predict_stride)]     # the first state of every predicted window
    leads = np.arange(1, cfg.predict_steps + 1) * (times[1] - times[0])
    batches, start = [], 0
    for prediction in predictions:                                                   # (b, v, steps, H, W) each
        b = prediction.shape[0]
        x = rearrange(prediction, "b v t h w -> v b t h w")
        batches.append(dataset.to_xarray(x, time=inits[start: start + b], prediction_timedelta=leads).astype("float32"))
        start += b
    ds = xr.concat(batches, dim="time")
    if path is not None:
        ds.chunk({"time": 16}).to_zarr(path, mode="w")
        ds = xr.open_zarr(path)
    return ds

In [ ]:
predictions = trainer.predict(module)
forecast = forecasts_to_xarray(module, predictions, path=FORECASTS / f"{RUN}.zarr")
forecast

The same two lines with a fresh `ForecastModule` from `configs/persistence.yaml` (no training, `Trainer.predict` only) give the persistence forecast of the year, `FORECASTS / "persistence.zarr"`.
Keep it: the checks of every part and the extension compare against it.

### 0.3 A sample

One initialisation, `Z500` and `T2M`: the forecast, the truth, and their difference, at 24 hours (step 4) and 72 hours (step 12).
The truth at a valid time is `era5` at `time + prediction_timedelta`; forecast and truth share one colour scale per variable, the difference gets a diverging one centred at zero.
Write the plot as a function of the forecast Dataset and the initialisation, so that every later run reuses it.
Which structures the forecast keeps between the two leads and which it loses, and where the difference concentrates, are what to look at.

### 0.4 The skill score against climatology (`utils/metrics.py`)

The forecast Dataset has the schema of lab 1, section 5, so the verification of lab 1 applies as it stands; the functions below are its pieces, on Datasets with the dimensions `(time, prediction_timedelta, latitude, longitude)`.

- `latitude_weights(latitude)`: cos(latitude) normalised to mean one, the relative area of a grid cell.
- `truth_at(era5, forecast)`: the stored fields at the valid times `time + prediction_timedelta` of the forecast, one `.sel` with the two-dimensional array of valid times (xarray's vectorised indexing).
  Rename the store's time axis before the `.sel`, so that the result keeps the forecast's `time` (the initialisation) and `prediction_timedelta` as its index coordinates.
- `climatology_at(clim, forecast)`: the (hour, dayofyear) climatology at the same valid times.
- `rmse_per_initialisation(forecast, truth)`: the area-weighted root mean squared error over latitude and longitude, one value per initialisation, lead, and variable.
- `rmse(forecast, truth)`: its mean over the initialisations, one value per lead and variable.
- `skill_score(rmse_forecast, rmse_reference)`: `1 - rmse_forecast / rmse_reference`; one is a perfect forecast, zero the reference, negative worse than the reference.

The climatology forecast is `climatology_at(clim, forecast)` itself, and the reference of the skill score.
Compute the RMSE of the model and of the climatology forecast for every variable and lead, the skill score from the two, and store it as `SKILL[RUN]`.
Plot the skill of `Z500` and `T2M` against the lead time in days with a line at zero, as a function that loops over `SKILL`, so that the models of the earlier parts stay on the axes when a run is added.
Where the curves cross zero, and how the two variables differ in that, is what to read off.

## 1. Loss weights: the weighted mean squared error

### 1.1 The module (`utils/loss_fn.py`)

The lecture listing: the squared error weighted per variable and per latitude row in one einsum, then the mean.

- `WeightedMSE(latitude, variable_weights=None, latitude_weighting=True)`: `latitude` is the latitude coordinate of the fields, `(h,)`; `per_latitude` is cos(latitude) normalised to mean one, or ones with `latitude_weighting=False`; `per_variable` is the given vector `(v,)` in the dataset's variable order, or ones (uniform, the setting of this part).
  `forward(prediction, target)` on `(b, v, h, w)`: the squared error, times `per_variable` over `v` and `per_latitude` over `h` in one `einsum`, then the mean over every axis.
  With uniform variable weights and no latitude weighting the value equals `MSE`.
- `ObjectiveConfig`: `name` is `"mse"` or `"weighted_mse"`; `kwargs` carries `latitude_weighting` and `variable_weights`, the latter a dictionary by variable name (`null` for uniform), which the LightningModule turns into the vector in the dataset's order.
- `ForecastModule.__init__` builds the training loss by `config.objective.name`, and gives it the latitude values of the dataset.
  The validation loss stays the plain `MSE`, whatever the training loss: `val/loss_step*` then measures the same quantity in every run, and the scores of section 0.4 are unweighted physical errors as well.

`configs/vit_area_mse.yaml` is section 0's run with `weighted_mse`, latitude weighting on, and uniform variable weights.

### 1.2 The loss curve at the end of every run (`utils/lightning_module.py`)

Every value passed to `self.log` is already in the run's `metrics.csv`; what is missing is the plot.

- `plot_metrics(log_dir, ax=None, label=None)`: reads `metrics.csv` from the log directory and plots `train/loss` against `step` from the rows that have it (a rolling mean over a few points on top of the raw values), and `val/loss_step1` against `step` as markers, on a logarithmic axis.
- `LossCurve(L.Callback)`: `on_train_end(trainer, module)` flushes the logger (`trainer.logger.save()`) and calls `plot_metrics` on `trainer.logger.log_dir`.
  Attach it to every `Trainer` from now on, `callbacks=[LossCurve()]`.

### 1.3 Train and compare

Train `vit_area_mse` with the cell below, then put its curve and section 0's (`plot_metrics` on `logs/vit_mse/version_0`) on one axis.
The two training lines are two different objectives; the validation markers are the same objective in both runs.
Whether the area weighting changes the training curve, the validation curve, or neither, at this training length, is the question.

### 1.4 Sample, skill, and a metric of your own

Steps 3 to 5 of every part: the sample plot of section 0.3 for the new run, its skill score added to `SKILL` and the plot of section 0.4 with both runs on it, and one more metric.
For the metric, go back to lab 1: sections 2, 5, and 8 defined more than the RMSE, and the model changed in a specific way here.
Choose the quantity of lab 1 that would show what the latitude weighting did to the forecast, and the region of the globe where it would show, and plot it for both runs.

Checks: `WeightedMSE` with `latitude_weighting=False` and no variable weights returns the same value as `MSE` on the same tensors; the latitude weights have mean one, and the ratio of the largest to the smallest is cos(2.8°) / cos(87.2°), about 20 on this grid; the module builds from the yaml unchanged.

In [ ]:
from utils.lightning_module import LossCurve, plot_metrics

RUN = "vit_area_mse"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=200, limit_val_batches=20, callbacks=[LossCurve()])
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
forecast = forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{RUN}.zarr")

## 2. Roll-out fine-tuning

### 2.1 The training step rolls out (`utils/lightning_module.py`)

The lecture listing: the training step applies the model a set number of times, its own output feeding the next step, and the loss is the mean over the steps; the gradients flow through the whole chain.

- `TrainerConfig` gains `train_rollout_steps` (the listing's `rollout_steps`; that name is taken by the validation roll-out since lab 2) and `pre_steps`.
- `training_step`: `steps` is 1 while `global_step < pre_steps` and `train_rollout_steps` after; from `batch[:, :, 0]`, apply the model `steps` times, add the training loss of model step `k` (`k = 0, ..., steps - 1`) against `batch[:, :, k + 1]`, and return the sum divided by `steps`.
  With `train_rollout_steps = 1` this is lab 2's training step.
- The training window has to hold the roll-out and its initial state: `sequence_length >= train_rollout_steps + 1` in `DatasetConfig`; assert it in `ForecastModule.__init__`.

### 2.2 Fine-tune from the checkpoint

`configs/vit_rollout.yaml` is the fine-tuning: four autoregressive steps in the training step, a window of five states, a learning rate ten times smaller, 300 steps.
The weights come from section 0's checkpoint and the datasets, optimiser, and schedule from the new config: `ForecastModule.load_from_checkpoint(CHECKPOINTS / "vit_mse.ckpt", config=Config.from_yaml(...))`, then `fit`.
The first logged loss is the roll-out error of the single-step model, the mean of its validation losses over steps 1 to 4, and the curve shows what the fine-tuning does to it.
The other way, the roll-out during training, is the same code with `pre_steps` set to the single-step phase and `max_steps` to the total (for example `vit_mse.yaml` with `pre_steps: 1500`, `train_rollout_steps: 4`, and `sequence_length: 5`); it is named here and not run.

### 2.3 Sample, skill, and a metric of your own

As in 1.4, with `SKILL` now holding three runs.
The roll-out trains the model on its own outputs; choose the quantity of lab 1 that shows a change in the forecasts over lead time, and the leads at which it shows.

Checks: `train_rollout_steps = 1` reproduces lab 2's training loss on a fixed batch; a training batch of the fine-tuning config has five states; the first `train/loss` of the fine-tuning is close to the mean of the section 0 model's `val/loss_step1` to `val/loss_step4`.

In [ ]:
RUN = "vit_rollout"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule.load_from_checkpoint(CHECKPOINTS / "vit_mse.ckpt", config=config)      # the weights of section 0, everything else from this config
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=100, limit_val_batches=20, callbacks=[LossCurve()])
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
forecast = forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{RUN}.zarr")

## 3. Heterogeneous variables

### 3.1 One embedding per variable (`utils/components.py`)

The lecture listing: the patch embedding as an EinMix whose weight carries the variable axis, so that no variable mixes with another at the first layer, and the tokens keep `(variable, dim)`; the unembedding is the mirror image.

- `NetworkConfig` gains `separable_embed` (default `False`).
- `ViT` with `separable_embed`: `to_tokens` maps `(b, v, H, W)` to `(b, h * w, v * dim)` with one weight per variable of shape `(hh, ww, dim)`.
  `to_fields` maps back with one weight per variable of shape `(dim, hh, ww)`.
  The positional embedding, the blocks, and the final norm run at the width `num_variables * dim`.
  Without the flag the ViT is lab 2's.
- `configs/vit_separable.yaml`: `separable_embed: true` and `dim: 16`, so the width is 9 times 16, 144, close to lab 2's 128.

Train it and put its curve next to section 0's; the parameter counts of the two models belong in the comparison.

### 3.2 A weight per variable in the training loss

`configs/vit_separable_weighted.yaml` trains the same model with `variable_weights` set by name, the surface weights and the pressure over 1000 hPa of the level variables after GraphCast, and `latitude_weighting: false`, so that one thing changes at a time.
The weights enter the training loss only.
The validation loss and the scores stay unweighted, so a variable with a small weight is not scored more leniently; it is trained less.
The training loss of this run is smaller by construction, the weights being at most one, so its training line does not compare to the unweighted runs; the validation markers do.
Compare the curves of 3.1 and 3.2.

### 3.3 Sample, skill, the map, and a metric of your own

As in 1.4, with `SKILL` holding five runs.
Then the skill per variable: for each run, a two-dimensional map with the variables on one axis and the lead time on the other, the skill score as the colour on a diverging scale centred at zero.
Which variables gained from the weights, which lost, and whether the separable embedding moved the map at all, are what to read off.
The model changed in how it treats the variables; choose a quantity of lab 1, per variable, that shows that.

Checks: the separable embedding has exactly `v * hh * ww * dim` weights and maps a state to a state of the same shape; with the weight of one variable set to zero, the gradient of the training loss with respect to that variable's prediction is zero; the persistence forecast of section 0.2 gives, at the first lead, the stored fields at the initialisation exactly, for every variable.

In [ ]:
RUN = "vit_separable"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=200, limit_val_batches=20, callbacks=[LossCurve()])
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
forecast = forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{RUN}.zarr")

In [ ]:
RUN = "vit_separable_weighted"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=200, limit_val_batches=20, callbacks=[LossCurve()])
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
forecast = forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{RUN}.zarr")

## Extension: the seasonal and diurnal cycle

### E.1 The error by hour of initialisation

The model of section 0 takes one state as its input; the hour of the day and the day of the year are not among its inputs.
The initialisations of `Trainer.predict` run through all four hours of the day, so the forecasts can be stratified by them: `rmse_per_initialisation`, grouped by the hour of the initialisation time and averaged within each group, gives `(hour, prediction_timedelta)` per variable.
Plot it for `T2M` of the section 0 run as a two-dimensional map, the hour of initialisation on one axis and the lead time on the other, the error as the colour, and the persistence forecast on the same map as the reference.
Whether the trained model's error depends on the hour of initialisation, more or less than persistence's does, and at which leads, is what to read off.

### E.2 The dataset returns the time (`utils/dataset.py`)

- `WeatherDataset.__getitem__(idx)` returns a pair: the window as before, and the time of its first state as one float32 number, the hours since 1 January of that year, `(dayofyear - 1) * 24 + hour`.
  A number rather than the `datetime64` itself, so that the DataLoader can batch it; the hour of the day is its remainder by 24, and the day of the year its quotient.
- The LightningModule unpacks the pair in its three steps, `states, time = batch`, and passes the time to the model.
  `forward(x, time=None)` and `forecast(x, steps, time=None)` take it as an optional argument; along a roll-out the time advances by the data step (six hours, read off the dataset) with every model step, in `forecast`, in the roll-out of the training step, and in the validation step.
  A model that ignores the time is unchanged by this.

### E.3 The time in the model (`utils/components.py`)

The lecture listing: sine and cosine features of a periodic coordinate at the harmonics of its period.

- `sincos_embedding(coordinate, period, dim)`: `dim // 2` frequencies, `2 * pi / period` times 1 to `dim // 2`; the angles as an einsum of the coordinate and the frequencies; the sines and the cosines concatenated on the last axis, `(..., dim)`.
- `NetworkConfig` gains `metadata_embed` (default `False`) and `dim_metadata` (8).
- `ViT.forward(x, time=None)` with `metadata_embed` on and a time given: the features of the time with period 24 hours and with period 365.25 days (in hours), `dim_metadata` each, concatenated to `(b, 2 * dim_metadata)`.
  One linear map takes them to the token width; `einops.repeat` broadcasts the result from `(b, d)` to `(b, n, d)` over the tokens; the result is added to the tokens after the patch embedding and the positions, before the blocks.
  Without the flag, or without a time, the ViT is the one before.

`configs/vit_time.yaml` is section 0's run with `metadata_embed: true`.

### E.4 Train, sample, skill, metric, and the map again

Train `vit_time`, then steps 3 to 5 as in 1.4, and the map of E.1 for the new run next to the old one.
Whether the dependence on the hour of initialisation changed, what happened to the T2M skill against the runs before, and whether anything changed for the other variables, are what to read off.

Checks: `sincos_embedding` at 0 hours and at 24 hours gives the same features for period 24; the ViT with the flag gives different outputs for the same state at 00 and 12 UTC and identical outputs for identical times; without the flag the time makes no difference; the first sample of the training dataset returns time 0.0 and the fifth 24.0.

In [ ]:
RUN = "vit_time"
config = Config.from_yaml(f"configs/{RUN}.yaml")
module = ForecastModule(config)
trainer = L.Trainer(max_steps=config.trainer.max_steps, accelerator="auto", logger=CSVLogger(LOGS, name=RUN), log_every_n_steps=10,
                    enable_checkpointing=False, val_check_interval=200, limit_val_batches=20, callbacks=[LossCurve()])
trainer.fit(module)
trainer.save_checkpoint(CHECKPOINTS / f"{RUN}.ckpt")
forecast = forecasts_to_xarray(module, trainer.predict(module), path=FORECASTS / f"{RUN}.zarr")